# Hotel Booking Cancellation — v2 cross-validation folds (group-aware, nested)

Every v2 model is tuned and compared on the same folds, so they are made once here and saved.
Only `data/processed/v2/train.csv` is read; the test file stays closed.

The folds group rows by booking profile, the same key the train/test split used: rows with identical
predictor values look the same to a model, so they must never sit on both the fitting and the
validation side of a fold.

In [1]:
import hashlib, json
from pathlib import Path

import pandas as pd
from sklearn.model_selection import StratifiedGroupKFold

## 1. Training rows and their profiles

In [2]:
out = Path('../../data/processed/v2')
train = pd.read_csv(out / 'train.csv', dtype={'agent': str})

X = train.drop(columns='is_canceled')
y = train['is_canceled']
groups = pd.util.hash_pandas_object(X, index=False).factorize()[0]

print(X.shape[1], 'predictors,', f'{len(X):,} rows, {groups.max() + 1:,} profiles,',
      'cancel rate', round(y.mean(), 4))
assert X.shape[1] == 25 and len(X) == 67974

25 predictors, 67,974 rows, 67,725 profiles, cancel rate 0.2775


## 2. Outer and inner folds

Outer: `StratifiedGroupKFold(5, shuffle, seed 42)` over the training rows. Inner: for each outer
fold, the rows outside it (the outer fitting part) get their own `StratifiedGroupKFold(3, shuffle,
seed 42)`. A row therefore has one outer fold and one inner fold per outer fold it is fitted in;
`inner_fold_k` is blank on rows whose outer fold is `k`.

In [3]:
SEED, N_OUTER, N_INNER = 42, 5, 3

folds = pd.DataFrame({'row_id': range(len(X)), 'outer_fold': -1})
outer = StratifiedGroupKFold(n_splits=N_OUTER, shuffle=True, random_state=SEED)
for k, (_, val_idx) in enumerate(outer.split(X, y, groups)):
    folds.loc[val_idx, 'outer_fold'] = k

inner = StratifiedGroupKFold(n_splits=N_INNER, shuffle=True, random_state=SEED)
for k in range(N_OUTER):
    fit_idx = folds.index[folds.outer_fold != k].to_numpy()
    col = pd.Series(pd.NA, index=folds.index, dtype='Int64')
    for j, (_, val_idx) in enumerate(inner.split(X.iloc[fit_idx], y.iloc[fit_idx], groups[fit_idx])):
        col.iloc[fit_idx[val_idx]] = j
    folds[f'inner_fold_{k}'] = col

assert (folds.outer_fold >= 0).all()
folds.head()

,row_id,outer_fold,inner_fold_0,inner_fold_1,inner_fold_2,inner_fold_3,inner_fold_4
0,0,4,2,1,0,0,<NA>
1,1,0,<NA>,0,0,0,2
2,2,0,<NA>,2,1,1,0
3,3,4,2,0,1,2,<NA>
4,4,2,0,0,<NA>,1,0


## 3. Checks: no profile crosses a fold, cancellation rate holds

In [4]:
rows = []
def check(level, fit_mask, val_mask):
    overlap = len(set(groups[fit_mask]) & set(groups[val_mask]))
    rows.append({'fold': level, 'fit_rows': int(fit_mask.sum()), 'val_rows': int(val_mask.sum()),
                 'fit_rate': round(y[fit_mask].mean(), 4), 'val_rate': round(y[val_mask].mean(), 4),
                 'profile_overlap': overlap})

for k in range(N_OUTER):
    val = (folds.outer_fold == k).to_numpy()
    check(f'outer {k}', ~val, val)
    inner_col = folds[f'inner_fold_{k}']
    assert inner_col[val].isna().all() and inner_col[~val].notna().all()
    for j in range(N_INNER):
        ival = (inner_col == j).fillna(False).to_numpy()
        check(f'outer {k} / inner {j}', ~val & ~ival, ival)

report = pd.DataFrame(rows)
assert (report.profile_overlap == 0).all()
assert (report[['fit_rate', 'val_rate']] - y.mean()).abs().max().max() < 0.01
report

,fold,fit_rows,val_rows,fit_rate,val_rate,profile_overlap
0,outer 0,54379,13595,0.2775,0.2775,0
1,outer 0 / inner 0,36253,18126,0.2775,0.2774,0
2,outer 0 / inner 1,36253,18126,0.2775,0.2774,0
3,outer 0 / inner 2,36252,18127,0.2774,0.2775,0
4,outer 1,54380,13594,0.2775,0.2775,0
5,outer 1 / inner 0,36253,18127,0.2775,0.2774,0
6,outer 1 / inner 1,36253,18127,0.2774,0.2775,0
7,outer 1 / inner 2,36254,18126,0.2775,0.2774,0
8,outer 2,54379,13595,0.2775,0.2775,0
9,outer 2 / inner 0,36253,18126,0.2775,0.2774,0


## 4. Save the folds and extend the manifest

One row per training row, in `train.csv` order (`row_id` is the 0-based row position). The hash and
fold settings go into the existing manifest next to the split.

In [5]:
folds.to_csv(out / 'cv_folds.csv', index=False)

manifest_path = out / 'manifest.json'
manifest = json.loads(manifest_path.read_text())
assert hashlib.sha256((out / 'train.csv').read_bytes()).hexdigest() == manifest['files']['train.csv']

manifest['cv_folds'] = {
    'file': 'cv_folds.csv',
    'built_from': 'train.csv',
    'row_id': '0-based row position in train.csv',
    'group_key': manifest['group_key'],
    'outer': f'StratifiedGroupKFold(n_splits={N_OUTER}, shuffle=True, random_state={SEED}) on all training rows',
    'inner': f'StratifiedGroupKFold(n_splits={N_INNER}, shuffle=True, random_state={SEED}) on the rows outside each outer fold; '
             'column inner_fold_k, blank where outer_fold == k',
    'max_profile_overlap': int(report.profile_overlap.max()),
    'cancel_rate_range': [float(report[['fit_rate', 'val_rate']].min().min()),
                          float(report[['fit_rate', 'val_rate']].max().max())],
}
manifest['files']['cv_folds.csv'] = hashlib.sha256((out / 'cv_folds.csv').read_bytes()).hexdigest()
manifest_path.write_text(json.dumps(manifest, indent=2))
print(json.dumps(manifest['cv_folds'], indent=2), manifest['files']['cv_folds.csv'])

{
  "file": "cv_folds.csv",
  "built_from": "train.csv",
  "row_id": "0-based row position in train.csv",
  "group_key": "exact match on all predictor columns (every column except is_canceled), agent as text",
  "outer": "StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42) on all training rows",
  "inner": "StratifiedGroupKFold(n_splits=3, shuffle=True, random_state=42) on the rows outside each outer fold; column inner_fold_k, blank where outer_fold == k",
  "max_profile_overlap": 0,
  "cancel_rate_range": [
    0.2774,
    0.2775
  ]
} 2e3c4134f37a0c36b83e44bb1d747b62277a88161a89be362f5997a630ae961d


Model notebooks read `cv_folds.csv` next to `train.csv`: outer folds for honest comparison between
models, `inner_fold_k` for tuning inside outer fold `k`.